In [0]:
%run ../delta_function

In [0]:
%run ./env

In [0]:
#bibliothèques à importer
import pandas as pd 
from pyspark.sql import functions as F
from functools import reduce
from pyspark.sql.utils import AnalysisException
from pyspark.sql import Window
from pyspark.sql.functions import current_timestamp
from pyspark.sql.types import TimestampType, StructField, DecimalType
from pyspark.sql.functions import to_date
import pyspark.sql.utils;
from pyspark.sql.types import StructType, StringType;
from pyspark.sql.functions import concat, lit, col, upper, max, when, size
from pyspark.sql.functions import udf
from pyspark.sql.types import IntegerType
from datetime import datetime, timedelta
from pyspark.sql.functions import regexp_replace
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql.functions import concat, lit, coalesce
from pyspark.sql.functions import col, expr
from pyspark.sql.functions import row_number
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from pyspark.sql.functions import concat, lit
from pyspark.sql.functions import concat_ws
from pyspark.sql.functions import col, date_trunc
from pyspark.sql.functions import first, col
spark.conf.set("spark.sql.execution.arrow.enabled", "true")

In [0]:
if current_environment =='preprd': 
    source_catalog = f"""ext_mal_psql_maite_vision_board_test.public"""    
else:
    source_catalog = f"""ext_mal_psql_maite_vision_board_{current_environment}.public"""

In [0]:
source_mal_maite = f"""mal_maite_{current_environment}"""

In [0]:
source_mal_maite_bi = f"""mal_maite_bi_{current_environment}"""

In [0]:
catalog_mal_maite = f"""mal_maite_"""
catalog_mal_maite_gold = f"""_{current_environment}.gold"""

In [0]:
# Construire les tables classiques avec le bon schema
batches_production_planning = spark.table(f"{source_catalog}.batches_production_planning")
recomendations = spark.table(f"{source_catalog}.recommendations")
processes_activities = spark.table(f"{source_catalog}.processes_activities")
parameters_localizations = spark.table(f"{source_catalog}.parameters_localizations")
batches = spark.table(f"{source_catalog}.batches")
production_line = spark.table(f"{source_catalog}.plants_production_lines")
manual_entries = spark.table(f"{source_catalog}.manual_entries")
parameters_variables = spark.table(f"{source_catalog}.parameters_variables")

# table parameters process activites pour filtrer les localisations attendues
parameters_process_activities_st2 = spark.table(f"{source_mal_maite}.strasbourg2.parameters_process_activities")
parameters_process_activities_ng1 = spark.table(f"{source_mal_maite}.nogent1.parameters_process_activities")
parameters_process_activities_ng2 = spark.table(f"{source_mal_maite}.nogent2.parameters_process_activities")
parameters_process_activities_ro1 = spark.table(f"{source_mal_maite}.rouen1.parameters_process_activities")
parameters_process_activities_pr1 = spark.table(f"{source_mal_maite}.prouvy1.parameters_process_activities")
parameters_process_activities_pl1 = spark.table(f"{source_mal_maite}.polisy1.parameters_process_activities")
parameters_process_activities_bu1 = spark.table(f"{source_mal_maite}.buzau1.parameters_process_activities")
parameters_process_activities_bo1 = spark.table(f"{source_mal_maite}.bolelemi1.parameters_process_activities")


# Ajout des tables "mal_maite_dev.inference_monitoring_candidates"
inference_monitoring_candidates_rouen1 = spark.table(f"mal_maite_{current_environment}.rouen1.inference_monitoring_candidates") 
inference_monitoring_candidates_strasbourg2 = spark.table(f"mal_maite_{current_environment}.strasbourg2.inference_monitoring_candidates")
inference_monitoring_candidates_prouvy1 = spark.table(f"mal_maite_{current_environment}.prouvy1.inference_monitoring_candidates")
inference_monitoring_candidates_polisy1 = spark.table(f"mal_maite_{current_environment}.polisy1.inference_monitoring_candidates")
#inference_monitoring_candidates_pithivier6 = spark.table(f"mal_maite_{current_environment}.pithiviers6.inference_monitoring_candidates")
inference_monitoring_candidates_nogent2 = spark.table(f"mal_maite_{current_environment}.nogent2.inference_monitoring_candidates")
inference_monitoring_candidates_nogent1 = spark.table(f"mal_maite_{current_environment}.nogent1.inference_monitoring_candidates")
inference_monitoring_candidates_buzau1 = spark.table(f"mal_maite_{current_environment}.buzau1.inference_monitoring_candidates")
inference_monitoring_candidates_bolelemi1 = spark.table(f"mal_maite_{current_environment}.bolelemi1.inference_monitoring_candidates")

# table monitoring DE
ascendance_st2 = spark.table(f"mal_maite_strasbourg2_{current_environment}.monitoring.historian_ancestry")
ascendance_ng1 = spark.table(f"mal_maite_nogent1_{current_environment}.monitoring.historian_ancestry")
ascendance_ng2 = spark.table(f"mal_maite_nogent2_{current_environment}.monitoring.historian_ancestry")
ascendance_ro1 = spark.table(f"mal_maite_rouen1_{current_environment}.monitoring.historian_ancestry")
ascendance_pr1 = spark.table(f"mal_maite_prouvy1_{current_environment}.monitoring.historian_ancestry")
ascendance_po1 = spark.table(f"mal_maite_polisy1_{current_environment}.monitoring.historian_ancestry")
#ascendance_pi6 = spark.table(f"mal_maite_pithiviers6_{current_environment}.monitoring.historian_ancestry")
ascendance_bu1 = spark.table(f"mal_maite_buzau1_{current_environment}.monitoring.historian_ancestry")
ascendance_bo1 = spark.table(f"mal_maite_bolelemi1_{current_environment}.monitoring.historian_ancestry")

missing_automatic_measures_st2 = spark.table(f"mal_maite_strasbourg2_{current_environment}.monitoring.missing_automatic_measures")
missing_automatic_measures_ng1 = spark.table(f"mal_maite_nogent1_{current_environment}.monitoring.missing_automatic_measures")
missing_automatic_measures_ng2 = spark.table(f"mal_maite_nogent2_{current_environment}.monitoring.missing_automatic_measures")
missing_automatic_measures_ro1 = spark.table(f"mal_maite_rouen1_{current_environment}.monitoring.missing_automatic_measures")
missing_automatic_measures_pr1 = spark.table(f"mal_maite_prouvy1_{current_environment}.monitoring.missing_automatic_measures")
missing_automatic_measures_po1 = spark.table(f"mal_maite_polisy1_{current_environment}.monitoring.missing_automatic_measures")
#missing_automatic_measures_pi6 = spark.table(f"mal_maite_pithiviers6_{current_environment}.monitoring.missing_automatic_measures")
missing_automatic_measures_bu1 = spark.table(f"mal_maite_buzau1_{current_environment}.monitoring.missing_automatic_measures")
missing_automatic_measures_bo1 = spark.table(f"mal_maite_bolelemi1_{current_environment}.monitoring.missing_automatic_measures")


# table de localisations old et nouvelle stack pour récupérer les prd_cell et workshop de chaque batch_id
new_stack_localisation_pr1 = spark.table(f"{catalog_mal_maite}prouvy1{catalog_mal_maite_gold}.localization_events")
new_stack_localisation_ng1 = spark.table(f"{catalog_mal_maite}nogent1{catalog_mal_maite_gold}.localization_events")
new_stack_localisation_ng2 = spark.table(f"{catalog_mal_maite}nogent2{catalog_mal_maite_gold}.localization_events")
new_stack_localisation_ro1 = spark.table(f"{catalog_mal_maite}rouen1{catalog_mal_maite_gold}.localization_events")
new_stack_localisation_st2 = spark.table(f"{catalog_mal_maite}strasbourg2{catalog_mal_maite_gold}.localization_events")
new_stack_localisation_po1 = spark.table(f"{catalog_mal_maite}polisy1{catalog_mal_maite_gold}.localization_events")
#new_stack_localisation_pi6 = spark.table(f"{catalog_mal_maite}pithiviers6{catalog_mal_maite_gold}.localization_events")
new_stack_localisation_bu1 = spark.table(f"{catalog_mal_maite}buzau1{catalog_mal_maite_gold}.localization_events")
new_stack_localisation_bo1 = spark.table(f"{catalog_mal_maite}bolelemi1{catalog_mal_maite_gold}.localization_events")

# tables intermédiaires missing values for asset
missing_asset_measure_ng1 = spark.table(f"mal_maite_nogent1_{current_environment}.monitoring.missing_measures_tags") 
missing_asset_measure_ng2 = spark.table(f"mal_maite_nogent2_{current_environment}.monitoring.missing_measures_tags")
missing_asset_measure_po1 = spark.table(f"mal_maite_polisy1_{current_environment}.monitoring.missing_measures_tags")
missing_asset_measure_st2 = spark.table(f"mal_maite_strasbourg2_{current_environment}.monitoring.missing_measures_tags")
missing_asset_measure_pr1 = spark.table(f"mal_maite_prouvy1_{current_environment}.monitoring.missing_measures_tags")
missing_asset_measure_ro1 = spark.table(f"mal_maite_rouen1_{current_environment}.monitoring.missing_measures_tags")
#missing_asset_measure_pi6 = spark.table(f"mal_maite_pithiviers6_{current_environment}.monitoring.missing_measures_tags")
missing_asset_measure_bu1 = spark.table(f"mal_maite_buzau1_{current_environment}.monitoring.missing_measures_tags")
missing_asset_measure_bo1 = spark.table(f"mal_maite_bolelemi1_{current_environment}.monitoring.missing_measures_tags")

# DM-5578
# la table inference_candidate se base sur la master table pour savoir si des valeurs automatiques sont manquantes ou non. 
# Cependant la master table est elle-même alimentée par la table measurement_pivot des DE. 
# Problématique : 
# On constate que sur certaines reco, les valeurs automatiques manquantes mentionnées dans la table inference_candidate car l’ingestion entre measurement_pivot et la master table n’a pas fonctionné

measurement_pivot_ng1 = spark.table(f"mal_maite_nogent1_{current_environment}.gold.measurements_pivot") 
measurement_pivot_ng2 = spark.table(f"mal_maite_nogent2_{current_environment}.gold.measurements_pivot")
measurement_pivot_po1 = spark.table(f"mal_maite_polisy1_{current_environment}.gold.measurements_pivot")
measurement_pivot_st2 = spark.table(f"mal_maite_strasbourg2_{current_environment}.gold.measurements_pivot")
measurement_pivot_pr1 = spark.table(f"mal_maite_prouvy1_{current_environment}.gold.measurements_pivot")
measurement_pivot_ro1 = spark.table(f"mal_maite_rouen1_{current_environment}.gold.measurements_pivot")
measurement_pivot_bu1 = spark.table(f"mal_maite_buzau1_{current_environment}.gold.measurements_pivot")
measurement_pivot_bo1 = spark.table(f"mal_maite_bolelemi1_{current_environment}.gold.measurements_pivot")